# Análisis de accesibilidad a servicios de salud en Sonora

Pregunta del proyecto: ¿Dónde se concentran las poblaciones de Sonora con menor accesibilidad geográfica a servicios de salud?

Este notebook responde la pregunta con la tabla final del pipeline (`data/processed/accesibilidad_localidades.parquet`): una fila por localidad con el tiempo de traslado por carretera al establecimiento de 1er y de 2do nivel más cercano, y las variables de vulnerabilidad del ITER.

Qué se revisa:
1. Qué tan completos están los tiempos medidos con SAKBÉ.
2. Cuánta población hay en cada rango de tiempo.
3. En qué municipios se concentra la población con acceso comprometido.
4. Qué localidades son prioritarias.
5. A qué grupos vulnerables afecta.
6. Qué tanto cambian los resultados con el umbral.

Los cálculos están en `src/health_data/analisis.py`, que tiene pruebas en `tests/test_analisis.py`. Este notebook solo los aplica y los grafica.

**Estado de los datos (18 de octubre de 2026):** el ruteo con SAKBÉ está completo. 7,213 de las 7,297 localidades tienen ambos tiempos medidos, el 99.99 % de la población. Las 84 restantes no tienen ruta SAKBÉ en algún nivel (56 no tienen carretera cercana y en 28 SAKBÉ no ubica a sus centros de salud candidatos) y usan el respaldo con CONEVAL en ese nivel (sección 1). Se excluyen 3 localidades de la Isla del Tiburón (16 habitantes), sin acceso por carretera. Si cambian los datos, ejecutar `make data` y volver a correr este notebook.

- Fuentes: INEGI (ITER 2020, Marco Geoestadístico, API de Ruteo SAKBÉ v3.1), Secretaría de Salud (CLUES) y CONEVAL (GACP 2020).
- Los tiempos de SAKBÉ son para automóvil en condiciones óptimas. Fuente: INEGI, API de Ruteo.

In [ ]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from health_data import analisis  # noqa: E402
from health_data.config import ARCHIVO_TABLA_FINAL, cargar_configuracion  # noqa: E402
from health_data.processing import cartografia  # noqa: E402
from health_data.visualization import mapas  # noqa: E402

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.1f}".format)
pio.templates.default = "plotly_white"
pio.renderers.default = "plotly_mimetype"

DIR_FIGURAS = ROOT / "reports" / "figures"
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)

UMBRALES = cargar_configuracion().umbrales
U1 = analisis.umbral_critico("primer", UMBRALES)
U2 = analisis.umbral_critico("segundo", UMBRALES)
AZUL, NARANJA, ROJO, GRIS = "#2a78d6", "#eb6834", "#B00020", "#8C8C8C"


def guardar(figura: go.Figure, nombre: str) -> go.Figure:
    """Guarda la figura como HTML interactivo en reports/figures."""
    figura.write_html(DIR_FIGURAS / f"{nombre}.html", include_plotlyjs="cdn")
    return figura


print(f"Umbral crítico: 1er nivel > {U1:.0f} min · 2do nivel > {U2:.0f} min")

## 1. Datos y cobertura de las rutas

Una localidad tiene **acceso comprometido** si supera el umbral crítico de 1er nivel o el de 2do nivel. Los umbrales vienen de `config/fuentes_health.yaml`.

El umbral de 2do nivel es de 120 min por decisión del equipo: quien necesita un hospital suele llegar en una condición en la que cada minuto cuenta, así que se prefiere un umbral estricto. Ese tiempo solo cubre el traslado; el tiempo dentro del establecimiento se analizará después.

Cifras generales del estado y qué parte de ellas se apoya en rutas medidas con SAKBÉ.

In [ ]:
tabla = pd.read_parquet(ARCHIVO_TABLA_FINAL)
resumen = analisis.resumen_estatal(tabla, umbrales=UMBRALES)
print(f"Tabla: {ARCHIVO_TABLA_FINAL.name} · {len(tabla):,} localidades · consultada el {date.today()}")
pd.Series(resumen).to_frame("valor")

Origen de los tiempos por localidad (`routing_status`): `ok` si ambos niveles se midieron con SAKBÉ, `parcial` si solo uno, `sin_ruta` si SAKBÉ no encontró carretera cercana y `estimado` si aún no se consulta. Por nivel, `primary_method` y `secondary_method` indican si el tiempo viene de SAKBÉ (`sakbe`), del respaldo con CONEVAL (`coneval`) o de la estimación por línea recta (`estimado`).

In [ ]:
origen = tabla.groupby("routing_status").agg(localidades=("population", "size"), poblacion=("population", "sum"))
origen["pct_localidades"] = 100 * origen["localidades"] / origen["localidades"].sum()
origen["pct_poblacion"] = 100 * origen["poblacion"] / origen["poblacion"].sum()
origen.sort_values("poblacion", ascending=False)

Las rutas se consultaron de la localidad más poblada a la menos poblada. Las que no tienen ruta SAKBÉ son pequeñas y están lejos de cualquier carretera pavimentada.

Para ellas se usa el **respaldo con CONEVAL**: el tiempo típico (mediana) medido con SAKBÉ en las localidades del mismo municipio y con el mismo grado de aislamiento según CONEVAL (tiempo al centro de servicios). Si en el municipio son menos de 5, se usa el de todo el estado. Comparado con las rutas reales en localidades parecidas (a 2 km o más de carretera pavimentada), acierta si la localidad supera el umbral en ~91 % de los casos para el hospital y ~84 % para el centro de salud, contra ~72 % de la estimación por línea recta.

## 2. Población por rango de tiempo de traslado

Localidades, población y grupos vulnerables en cada rango de tiempo. Los rangos usan los umbrales del proyecto.

In [ ]:
rangos_1 = analisis.poblacion_por_rango(tabla, "primer", UMBRALES)
rangos_1

In [ ]:
rangos_2 = analisis.poblacion_por_rango(tabla, "segundo", UMBRALES)
rangos_2

In [ ]:
fig = go.Figure()
for nombre, rangos, color in [("1er nivel", rangos_1, AZUL), ("2do nivel", rangos_2, NARANJA)]:
    fig.add_bar(
        x=rangos.index.astype(str),
        y=rangos["poblacion"],
        name=nombre,
        marker_color=color,
        customdata=rangos[["pct_poblacion", "localidades"]],
        hovertemplate="%{x}<br>%{y:,} personas (%{customdata[0]:.1f} %)<br>%{customdata[1]:,} localidades<extra>"
        + nombre
        + "</extra>",
    )
fig.update_layout(
    title="Población por tiempo de traslado al establecimiento más cercano",
    xaxis_title="Tiempo de traslado",
    yaxis_title="Personas (escala logarítmica)",
    yaxis_type="log",
    barmode="group",
)
guardar(fig, "poblacion_por_rango_tiempo")

La escala es logarítmica porque casi toda la población vive cerca de un establecimiento y los rangos altos tienen pocas personas. Los rangos no son iguales en los dos niveles: para el hospital (2do nivel) se espera un traslado más largo.

## 3. ¿Dónde? Municipios con acceso comprometido

Indicadores por municipio, ordenados por población con acceso comprometido. Los tiempos promedio se **ponderan por población**: reflejan el tiempo que enfrenta una persona típica del municipio, no una localidad típica (la mayoría de las localidades tiene menos de 10 habitantes).

`pct_localidades_sakbe` indica qué parte de las localidades del municipio tiene ambos tiempos medidos con SAKBÉ.

In [ ]:
municipios = analisis.resumen_municipal(tabla, umbrales=UMBRALES)
columnas = [
    "municipio",
    "poblacion",
    "poblacion_comprometida",
    "pct_poblacion_comprometida",
    "poblacion_primer_critico",
    "poblacion_segundo_critico",
    "tiempo_primer_ponderado",
    "tiempo_segundo_ponderado",
    "pct_localidades_sakbe",
]
municipios.loc[municipios["poblacion_comprometida"] > 0, columnas]

In [ ]:
top = municipios[municipios["poblacion_comprometida"] > 0].head(15).iloc[::-1]
fig = go.Figure(
    go.Bar(
        x=top["poblacion_comprometida"],
        y=top["municipio"],
        orientation="h",
        marker=dict(color=top["pct_poblacion_comprometida"], colorscale="OrRd", colorbar=dict(title="% del municipio")),
        customdata=top[["pct_poblacion_comprometida", "tiempo_segundo_ponderado"]],
        hovertemplate="<b>%{y}</b><br>%{x:,} personas (%{customdata[0]:.1f} % del municipio)"
        "<br>Tiempo promedio al hospital: %{customdata[1]:.0f} min<extra></extra>",
    )
)
fig.update_layout(
    title=f"Municipios con más población con acceso comprometido (1er nivel > {U1:.0f} o 2do nivel > {U2:.0f} min)",
    xaxis_title="Personas",
    height=520,
)
guardar(fig, "municipios_poblacion_comprometida")

El tamaño de la barra es la cantidad de personas y el color es qué parte del municipio representan. Un municipio puede tener pocas personas afectadas pero ser casi todo su territorio, o al revés.

Mapa del tiempo promedio ponderado al hospital (2do nivel) por municipio.

In [ ]:
geojson = cartografia.cargar_geojson(cartografia.NOMBRE_MUNICIPIOS_SONORA)
fig = mapas.mapa_municipios(
    municipios,
    geojson,
    columna="tiempo_segundo_ponderado",
    columna_nombre="municipio",
    titulo="Tiempo promedio al establecimiento de 2do nivel (ponderado por población)",
)
guardar(fig, "mapa_municipios_tiempo_2do_nivel")

## 4. Localidades prioritarias

Localidades con acceso comprometido, de mayor a menor población. Las columnas `*_method` indican si el tiempo se midió con SAKBÉ (`sakbe`), se asignó con CONEVAL (`coneval`) o se estimó (`estimado`).

In [ ]:
prioritarias = analisis.localidades_prioritarias(tabla, n=None, umbrales=UMBRALES)
print(f"{len(prioritarias):,} localidades con acceso comprometido · {prioritarias['population'].sum():,} personas")
prioritarias.head(25)

In [ ]:
datos_mapa = tabla.assign(comprometida=analisis.acceso_comprometido(tabla, umbrales=UMBRALES))
fig = px.scatter_mapbox(
    datos_mapa.sort_values("comprometida"),
    lat="latitude",
    lon="longitude",
    color=datos_mapa.sort_values("comprometida")["comprometida"].map({True: "Acceso comprometido", False: "Dentro del umbral"}),
    color_discrete_map={"Acceso comprometido": ROJO, "Dentro del umbral": GRIS},
    size=datos_mapa.sort_values("comprometida")["population"].clip(lower=1) ** 0.5,
    size_max=18,
    hover_name="locality_name",
    hover_data={"municipality_name": True, "population": ":,", "primary_time_min": ":.0f", "secondary_time_min": ":.0f", "latitude": False, "longitude": False},
    mapbox_style="white-bg",
    zoom=5.6,
    center=dict(lat=29.3, lon=-110.7),
    height=650,
    title="Localidades de Sonora según acceso a servicios de salud (tamaño: población)",
)
fig.update_layout(legend_title_text="", mapbox=mapas.estilo_mapa_base())
guardar(fig, "mapa_localidades_acceso_comprometido")

## 5. Grupos vulnerables con acceso comprometido

Personas de cada grupo que viven en localidades con acceso comprometido, frente al total del estado. ITER no publica estas variables en las localidades confidenciales (de una o dos viviendas), así que los totales cubren a la población indicada en `pct_poblacion_con_datos_vulnerabilidad` de la sección 1.

La población sin automóvil ni motocicleta es una estimación: población de la localidad × porcentaje de viviendas sin vehículo.

In [ ]:
grupos = analisis.grupos_vulnerables(tabla, umbrales=UMBRALES)
grupos

In [ ]:
fig = mapas.barras_grupos_vulnerables(grupos)
guardar(fig, "grupos_vulnerables_acceso_comprometido")

Si la barra de un grupo es más larga que la de la población total, ese grupo está sobrerrepresentado entre quienes tienen acceso comprometido.

## 6. Sensibilidad al umbral de 2do nivel

El umbral crítico de 2do nivel cambia mucho cuántas personas quedan con acceso comprometido. Población afectada con distintos umbrales, manteniendo fijo el de 1er nivel. La línea punteada marca el umbral del proyecto (120 min).

In [ ]:
sensibilidad = pd.DataFrame(
    [
        {
            "umbral_2do_nivel_min": u,
            "localidades": int(analisis.acceso_comprometido(tabla, umbral_segundo=u, umbrales=UMBRALES).sum()),
            "poblacion": int(tabla.loc[analisis.acceso_comprometido(tabla, umbral_segundo=u, umbrales=UMBRALES), "population"].sum()),
        }
        for u in [60, 90, 120, 150, 180, 240]
    ]
)
sensibilidad

In [ ]:
fig = px.line(
    sensibilidad,
    x="umbral_2do_nivel_min",
    y="poblacion",
    markers=True,
    title="Población con acceso comprometido según el umbral de 2do nivel",
    labels={"umbral_2do_nivel_min": "Umbral de 2do nivel (minutos)", "poblacion": "Personas"},
)
fig.add_vline(x=U2, line_dash="dash", line_color=ROJO, annotation_text=f"Umbral del proyecto ({U2:.0f} min)")
fig.update_traces(line_color=AZUL)
guardar(fig, "sensibilidad_umbral_2do_nivel")

## 7. Resumen

Cifras clave calculadas con la tabla actual. Se actualizan al volver a correr el notebook.

In [ ]:
peor_1 = municipios.sort_values("tiempo_primer_ponderado", ascending=False).iloc[0]
peor_2 = municipios.sort_values("tiempo_segundo_ponderado", ascending=False).iloc[0]
print(f"Población con acceso comprometido: {resumen['poblacion_comprometida']:,} personas "
      f"({resumen['pct_poblacion_comprometida']:.2f} % del estado) en {resumen['localidades_comprometidas']:,} localidades.")
print(f"Municipios con alguna localidad comprometida: {(municipios['poblacion_comprometida'] > 0).sum()} de {len(municipios)}.")
print("Municipios con más personas afectadas: " + ", ".join(
    f"{m} ({p:,})" for m, p in municipios.head(5)[["municipio", "poblacion_comprometida"]].itertuples(index=False)))
print(f"Mayor tiempo promedio al 1er nivel: {peor_1['municipio']} ({peor_1['tiempo_primer_ponderado']:.0f} min).")
print(f"Mayor tiempo promedio al 2do nivel: {peor_2['municipio']} ({peor_2['tiempo_segundo_ponderado']:.0f} min).")
print(f"Tiempos medidos con SAKBÉ: {resumen['pct_localidades_sakbe']:.1f} % de las localidades, "
      f"{resumen['pct_poblacion_sakbe']:.1f} % de la población.")

## 8. Conclusiones y limitaciones

Con datos del 18 de octubre de 2026: rutas medidas con SAKBÉ para 7,213 de 7,297 localidades, el 99.99 % de la población. Umbrales del proyecto: más de 60 min a un centro de salud o más de 120 min a un hospital.

**Conclusiones:**

1. **El acceso al 1er nivel es casi universal.** El 99.45 % de la población vive a 30 min o menos de un centro de salud. Solo 3,158 personas (0.11 %) están a más de 60 min.
2. **La brecha está en el hospital.** 20,520 personas (0.70 %) viven a más de 120 min de un establecimiento de 2do nivel, y 9,822 a más de 3 horas. De las 21,567 personas con acceso comprometido, 18,409 (85 %) lo tienen solo por el hospital: tienen un centro de salud cerca, pero el hospital lejos.
3. **Se concentra en pocos municipios de la sierra y del desierto.** 53 de los 72 municipios tienen alguna localidad con acceso comprometido, pero cinco reúnen el 58 % de las personas afectadas: Yécora (4,793), Pitiquito (3,371), Sáric (2,042), Álamos (1,211) y Suaqui Grande (1,114). En Yécora, Suaqui Grande, Onavas y San Javier es toda la población del municipio, y en Sáric el 99 %.
4. **No son solo ranchos aislados.** 40 localidades de 100 habitantes o más tienen acceso comprometido, entre ellas Puerto Libertad (2,750 habitantes, 172 min al hospital), Yécora (2,453; 182 min), Suaqui Grande (1,074; 187 min), Sásabe (977; 199 min) y Maycoba (658; 216 min). Todas tienen un centro de salud a pocos minutos: el problema es la distancia al hospital.
5. **Los grupos vulnerables están sobrerrepresentados.** Frente al 0.73 % de la población general, tienen acceso comprometido el 1.48 % de los hablantes de lengua indígena (el doble), el 1.09 % de quienes no tienen afiliación a servicios de salud, el 0.98 % de las personas de 60 años y más y el 0.91 % de las personas con discapacidad. Además, unas 7,659 personas con acceso comprometido viven en hogares sin automóvil ni motocicleta, y para ellas el tiempo real es mayor.
6. **El resultado depende del umbral de 2do nivel.** Con 120 min, el del proyecto, son 21,567 personas; con 90 min serían 39,840, y con 60 min, 99,333.
7. **La estimación por línea recta no sirve en las zonas alejadas.** Con todas las rutas medidas, SAKBÉ dio tiempos al hospital 24 % mayores que la estimación en la mediana, y 2.4 veces mayores en las localidades a más de 120 min. La estimación solo habría detectado 68 de las 635 localidades realmente críticas. Por eso las localidades sin carretera usan el respaldo con CONEVAL.

**Limitaciones:**
- Se mide accesibilidad geográfica potencial en automóvil en condiciones óptimas. No se miden transporte público, costo ni capacidad de los establecimientos.
- El tiempo es solo de traslado. Falta el tiempo dentro del establecimiento (espera y atención), que se analizará después y se sumaría al traslado.
- Una parte de la población no tiene automóvil (ver sección 5); para ella el tiempo real es mayor.
- 84 localidades (397 habitantes) no tienen ruta SAKBÉ en algún nivel y usan el respaldo con CONEVAL, que es una aproximación: acierta el lado del umbral en ~84–91 % de los casos.
- Se excluyen 3 localidades de la Isla del Tiburón (16 habitantes), sin acceso por carretera y que parecen campamentos de pesca.
- Los grupos vulnerables no incluyen a las localidades confidenciales de ITER (0.8 % de la población).
- El establecimiento más cercano se elige entre los 2 más cercanos en línea recta; en terrenos muy accidentados podría existir uno más rápido por carretera.